## XGBoost model

Import packages.

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Update config.

In [2]:
stock = "NVDA"
market = "QQQ"

period = "5y"
interval = "1d"

prediction_horizon = 5
return_threshold = 0.01

n_splits = 5

Fetch the data.

In [3]:
stock_data = yf.download(
    stock,
    period=period,
    interval=interval,
    auto_adjust=True,
    progress=False
)

market_data = yf.download(
    market,
    period=period,
    interval=interval,
    auto_adjust=True,
    progress=False
)

if stock_data.empty:
    raise ValueError(f"No data downloaded for {stock}")

if market_data.empty:
    raise ValueError(f"No data downloaded for {market}")

if isinstance(stock_data.columns, pd.MultiIndex):
    stock_data.columns = [col[0] for col in stock_data.columns]

if isinstance(market_data.columns, pd.MultiIndex):
    market_data.columns = [col[0] for col in market_data.columns]

Create features for the model.

In [4]:

df = pd.DataFrame(index=stock_data.index)

df["stock_close"] = stock_data["Close"]
df["market_close"] = market_data["Close"]

df["stock_return"] = stock_data["Close"].pct_change()
df["market_return"] = market_data["Close"].pct_change()

df["relative_strength"] = df["stock_return"] - df["market_return"]

df["momentum_5"] = stock_data["Close"] / stock_data["Close"].shift(5) - 1
df["momentum_20"] = stock_data["Close"] / stock_data["Close"].shift(20) - 1
df["momentum_60"] = stock_data["Close"] / stock_data["Close"].shift(60) - 1

df["market_momentum_5"] = market_data["Close"] / market_data["Close"].shift(5) - 1
df["market_momentum_20"] = market_data["Close"] / market_data["Close"].shift(20) - 1

df["volatility_10"] = df["stock_return"].rolling(10).std()
df["volatility_20"] = df["stock_return"].rolling(20).std()

df["volume_ratio"] = stock_data["Volume"] / stock_data["Volume"].rolling(20).mean()

df["ma_5"] = stock_data["Close"].rolling(5).mean()
df["ma_20"] = stock_data["Close"].rolling(20).mean()
df["ma_50"] = stock_data["Close"].rolling(50).mean()

df["ma_ratio_5_20"] = df["ma_5"] / df["ma_20"]
df["ma_ratio_20_50"] = df["ma_20"] / df["ma_50"]

df.tail()

,stock_close,market_close,stock_return,market_return,relative_strength,momentum_5,momentum_20,momentum_60,market_momentum_5,market_momentum_20,volatility_10,volatility_20,volume_ratio,ma_5,ma_20,ma_50,ma_ratio_5_20,ma_ratio_20_50
Date,,,,,,,,,,,,,,,,,,
2026-06-01,224.098816,742.739990,0.062612,0.006000,0.056612,0.041936,0.130562,0.223804,0.035120,0.101743,0.024467,0.026573,1.248741,215.191199,216.501171,200.031665,0.993949,1.082334
2026-06-02,222.560608,746.159973,-0.006864,0.004605,-0.011469,0.037047,0.122632,0.253133,0.021745,0.108905,0.024165,0.026707,1.112227,216.781345,217.716755,201.032899,0.995704,1.082991
2026-06-03,214.500000,744.210022,-0.036218,-0.002613,-0.033604,0.010113,0.092875,0.175810,0.020234,0.091841,0.026779,0.028146,0.913067,217.210843,218.628193,201.814188,0.993517,1.083314
2026-06-04,218.660004,740.609985,0.019394,-0.004837,0.024231,0.021773,0.053336,0.184861,0.006811,0.064447,0.027261,0.025546,0.964409,218.142728,219.181790,202.687467,0.995259,1.081378
2026-06-05,205.100006,705.059998,-0.062014,-0.048001,-0.014013,-0.027474,-0.029130,0.103795,-0.045035,0.014562,0.033117,0.029091,1.229272,216.983887,218.874101,203.220028,0.991364,1.077030


Create target variable.

In [5]:

df["future_return"] = (
    stock_data["Close"].shift(-prediction_horizon)
    / stock_data["Close"]
    - 1
)

df["target"] = (df["future_return"] > return_threshold).astype(int)

df = df.dropna()

df

,stock_close,market_close,stock_return,market_return,relative_strength,momentum_5,momentum_20,momentum_60,market_momentum_5,market_momentum_20,volatility_10,volatility_20,volume_ratio,ma_5,ma_20,ma_50,ma_ratio_5_20,ma_ratio_20_50,future_return,target
Date,,,,,,,,,,,,,,,,,,,,
2021-08-31,22.311779,369.142670,-0.013181,-0.000815,-0.012365,0.027346,0.129899,0.271019,0.014905,0.035822,0.027726,0.023787,0.881276,22.321583,20.733632,20.019553,1.076588,1.035669,-0.002055,0
2021-09-01,22.367590,369.754761,0.002501,0.001658,0.000843,0.010442,0.107081,0.286023,0.015422,0.036043,0.025192,0.023466,0.687734,22.367813,20.841806,20.090472,1.073219,1.037398,-0.011764,0
2021-09-02,22.322737,369.579803,-0.002005,-0.000473,-0.001532,0.015042,0.085427,0.290450,0.021427,0.029026,0.024419,0.023328,0.642828,22.433974,20.929649,20.157096,1.071875,1.038327,0.003662,0
2021-09-03,22.768278,370.716583,0.019959,0.003076,0.016883,0.009323,0.121822,0.311164,0.014706,0.036734,0.020603,0.023195,0.943595,22.476035,21.053273,20.229675,1.067579,1.040712,-0.030250,0
2021-09-07,22.587868,371.241180,-0.007924,0.001415,-0.009339,-0.000970,0.116827,0.271567,0.004865,0.036287,0.013651,0.023312,0.660611,22.471650,21.171415,20.302124,1.061415,1.042818,-0.018533,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-22,215.079330,717.539978,-0.019042,0.004241,-0.023283,-0.044337,0.033898,0.164702,0.012145,0.080828,0.025621,0.027372,1.040727,219.991605,214.495509,196.578689,1.025623,1.091143,0.041936,1
2026-05-26,214.609879,730.280029,-0.002183,0.017755,-0.019938,-0.033555,-0.008079,0.212662,0.034567,0.099439,0.024703,0.025872,1.150933,218.501340,214.408111,197.270084,1.019091,1.086876,0.037047,1
2026-05-27,212.352509,729.450012,-0.010518,-0.001137,-0.009382,-0.036308,-0.002674,0.165122,0.039799,0.109345,0.024669,0.025727,1.034452,216.901205,214.379644,197.857000,1.011762,1.083508,0.010113,1


Run modelling.

In [6]:
features = [
    "stock_return",
    "market_return",
    "relative_strength",
    "momentum_5",
    "momentum_20",
    "momentum_60",
    "market_momentum_5",
    "market_momentum_20",
    "volatility_10",
    "volatility_20",
    "volume_ratio",
    "ma_ratio_5_20",
    "ma_ratio_20_50",
]

X = df[features]
y = df["target"]

tscv = TimeSeriesSplit(n_splits=n_splits)

fold_results = []
all_test_predictions = []
all_test_probabilities = []
all_test_actuals = []
all_test_dates = []

print("\n================ TIME SERIES SPLIT RESULTS ================")

for fold, (train_index, test_index) in enumerate(tscv.split(X), start=1):
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    model = XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=2.0,
        gamma=0.1,
        min_child_weight=5,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, predictions)

    fold_results.append({
        "fold": fold,
        "train_start": X_train.index[0],
        "train_end": X_train.index[-1],
        "test_start": X_test.index[0],
        "test_end": X_test.index[-1],
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "accuracy": accuracy,
    })

    all_test_predictions.extend(predictions)
    all_test_probabilities.extend(probabilities)
    all_test_actuals.extend(y_test.values)
    all_test_dates.extend(X_test.index)

    print(f"\nFold {fold}")
    print("Train:", X_train.index[0], "to", X_train.index[-1])
    print("Test :", X_test.index[0], "to", X_test.index[-1])
    print("Accuracy:", accuracy)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, predictions))


================ TIME SERIES SPLIT RESULTS ================

Fold 1
Train: 2021-08-31 00:00:00 to 2022-06-16 00:00:00
Test : 2022-06-17 00:00:00 to 2023-03-31 00:00:00
Accuracy: 0.4797979797979798
Confusion Matrix:
[[73 15]
 [88 22]]

Fold 2
Train: 2021-08-31 00:00:00 to 2023-03-31 00:00:00
Test : 2023-04-03 00:00:00 to 2024-01-16 00:00:00
Accuracy: 0.48484848484848486
Confusion Matrix:
[[49 43]
 [59 47]]

Fold 3
Train: 2021-08-31 00:00:00 to 2024-01-16 00:00:00
Test : 2024-01-17 00:00:00 to 2024-10-28 00:00:00
Accuracy: 0.5202020202020202
Confusion Matrix:
[[45 40]
 [55 58]]

Fold 4
Train: 2021-08-31 00:00:00 to 2024-10-28 00:00:00
Test : 2024-10-29 00:00:00 to 2025-08-14 00:00:00
Accuracy: 0.51010101010101
Confusion Matrix:
[[44 48]
 [49 57]]

Fold 5
Train: 2021-08-31 00:00:00 to 2025-08-14 00:00:00
Test : 2025-08-15 00:00:00 to 2026-05-29 00:00:00
Accuracy: 0.4090909090909091
Confusion Matrix:
[[31 83]
 [34 50]]


Display overall metrics.

In [7]:
fold_results_df = pd.DataFrame(fold_results)

print("\n================ FOLD SUMMARY ================")
print(fold_results_df)

print("\nAverage fold accuracy:")
print(fold_results_df["accuracy"].mean())

print("\nOverall Confusion Matrix:")
print(confusion_matrix(all_test_actuals, all_test_predictions))

print("\nOverall Classification Report:")
print(classification_report(all_test_actuals, all_test_predictions))


================ FOLD SUMMARY ================
   fold train_start  train_end test_start   test_end  train_rows  test_rows  \
0     1  2021-08-31 2022-06-16 2022-06-17 2023-03-31         201        198   
1     2  2021-08-31 2023-03-31 2023-04-03 2024-01-16         399        198   
2     3  2021-08-31 2024-01-16 2024-01-17 2024-10-28         597        198   
3     4  2021-08-31 2024-10-28 2024-10-29 2025-08-14         795        198   
4     5  2021-08-31 2025-08-14 2025-08-15 2026-05-29         993        198   

   accuracy  
0  0.479798  
1  0.484848  
2  0.520202  
3  0.510101  
4  0.409091  

Average fold accuracy:
0.4808080808080808

Overall Confusion Matrix:
[[242 229]
 [285 234]]

Overall Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.51      0.48       471
           1       0.51      0.45      0.48       519

    accuracy                           0.48       990
   macro avg       0.48      0.48      0.48       

Recent out-of-sample predictions.

In [8]:
oos_predictions = pd.DataFrame({
    "date": all_test_dates,
    "actual_target": all_test_actuals,
    "predicted_probability": all_test_probabilities,
    "predicted_class": all_test_predictions,
})

oos_predictions = oos_predictions.set_index("date")

oos_predictions["stock_close"] = df.loc[oos_predictions.index, "stock_close"]
oos_predictions["future_return"] = df.loc[oos_predictions.index, "future_return"]

print("\n================ RECENT OUT-OF-SAMPLE PREDICTIONS ================")
print(oos_predictions.tail(15))


================ RECENT OUT-OF-SAMPLE PREDICTIONS ================
            actual_target  predicted_probability  predicted_class  \
date                                                                
2026-05-08              1               0.336167                0   
2026-05-11              1               0.416268                0   
2026-05-12              0               0.394919                0   
2026-05-13              0               0.603571                1   
2026-05-14              0               0.563609                1   
2026-05-15              0               0.554363                1   
2026-05-18              0               0.577343                1   
2026-05-19              0               0.555162                1   
2026-05-20              0               0.477840                0   
2026-05-21              0               0.383541                0   
2026-05-22              1               0.456239                0   
2026-05-26              1          

Train final model on all available data.

In [9]:
final_model = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    gamma=0.1,
    min_child_weight=5,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
)

final_model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0.1,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=5, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

Latest investment signal.

In [10]:
latest_row = X.tail(1)
latest_probability = final_model.predict_proba(latest_row)[0][1]
latest_date = df.index[-1]

print("\n================ LATEST INVESTMENT SIGNAL ================")
print("Latest date:", latest_date)
print("Stock:", stock)
print("Market benchmark:", market)
print("Probability of rising more than 1% over next 5 trading days:")
print(latest_probability)

if latest_probability >= 0.60:
    print("Signal: Bullish")
elif latest_probability <= 0.40:
    print("Signal: Bearish")
else:
    print("Signal: Neutral")


================ LATEST INVESTMENT SIGNAL ================
Latest date: 2026-05-29 00:00:00
Stock: NVDA
Market benchmark: QQQ
Probability of rising more than 1% over next 5 trading days:
0.26172614
Signal: Bearish


Feature importance from final model.

In [11]:
importance_df = pd.DataFrame({
    "feature": features,
    "importance": final_model.feature_importances_,
}).sort_values(by="importance", ascending=False)

print("\n================ FEATURE IMPORTANCE ================")
print(importance_df)


================ FEATURE IMPORTANCE ================
               feature  importance
4          momentum_20    0.095151
5          momentum_60    0.087997
9        volatility_20    0.087927
7   market_momentum_20    0.083469
12      ma_ratio_20_50    0.082802
6    market_momentum_5    0.078295
8        volatility_10    0.076440
11       ma_ratio_5_20    0.076230
2    relative_strength    0.071960
3           momentum_5    0.070555
10        volume_ratio    0.064384
0         stock_return    0.062743
1        market_return    0.062046
